# 🏗️ Structural Engineering AI - Training Gemma 2 9B (L4 GPU)

Notebook ini disiapkan khusus untuk Milestone **18 April**. 

**Fitur:**
1. Hybrid Dataset (SNI Theory + Structural Analysis CoT).
2. Optimized for L4 GPU (Google Colab).
3. Automatic Saving to Google Drive.

## 1. Persiapan Environment

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone Repository (Gunakan URL Repo Anda)
!git clone https://github.com/tech4iddev/COLAB_GEMMA_4.git /content/structural_ai
%cd /content/structural_ai/model_18_april

# 3. Install Unsloth & Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Sinkronisasi Data (Master Pipeline)
Gunakan cell ini untuk menggabungkan seluruh file Markdown SNI dan soal hitungan sintetis menjadi satu file training: `dataset_final_18_april.jsonl`.

In [ ]:
!python master_pipeline.py

### 🔍 Debug: Cek Kualitas Dataset
Pastikan data tidak kosong dan formatnya sudah benar.

In [ ]:
import json

dataset_path = "datasets/dataset_final_18_april.jsonl"
with open(dataset_path, 'r') as f:
    data = [json.loads(line) for line in f.readlines()]

print(f"Total Data: {len(data)}")

# Cek 1 contoh data Teori
print("\n--- Contoh Data Teori ---")
print(data[0])

# Cek 1 contoh data Analisa (Cari data yang punya 'Mu' atau 'kNm')
print("\n--- Contoh Data Analisa/Hitungan ---")
for item in data:
    if "kNm" in item['instruction'] or "Hitung" in item['instruction']:
        print(item)
        break

## 3. Proses Training
Menjalankan script training yang dioptimasi untuk L4.

In [ ]:
!python train_colab_L4.py

## 4. Testing & Inference (Debug Pasca-Training)
Mari kita coba apakah model sudah bisa menjawab soal hitungan.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma2-9b-structural-18april", # Folder hasil training
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

def ask_ai(instruction, input_text=""):
    prompt_style = """<start_of_turn>user
{} 
{}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer([prompt_style.format(instruction, input_text)], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
    return tokenizer.batch_decode(outputs)[0]

# TEST HITUNGAN
response = ask_ai("Hitung kapasitas tarik profil baja jika fy=250 MPa dan A=3000 mm2.")
print(response)

## 5. Simpan ke Google Drive
Langkah terakhir adalah memindahkan model ke Drive agar aman.

In [ ]:
import shutil
import os

source = "gemma2-9b-structural-18april"
dest = "/content/drive/MyDrive/Structural_AI_Models/18_April_Gemma2_9B"

if os.path.exists(source):
    if not os.path.exists(os.path.dirname(dest)):
        os.makedirs(os.path.dirname(dest))
    shutil.move(source, dest)
    print(f"✅ Model berhasil disimpan ke Drive: {dest}")